# Implementation of Deutsch algorithm using numpy

The Deutsch algorithm is one of the ﬁrst quantum algorithms which showed
quantum algorithms may be more eﬃcient than their classical counterparts.

## Import Packages

In [12]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np

from common.gates import H, I, tensor
from common.states import basis_state, probabilities
from common.oracles import (
    deutsch_oracle,
    DEUTSCH_FUNCTIONS
)
from common.visualization import (
    print_state,
    print_probabilities
)

## Prepare Initial State

In [13]:
# Initial state |01>
psi_initial = basis_state(0, 1)

print("Initial state:")
print_state(psi_initial)

Initial state:
(1.000+0.000j)|01>


## Hadamard transformation

Apply this transformation to get $|\psi_0\rangle$. This is like applying in first bit tensor in second bit

In [14]:
# Apply H ⊗ H
H_H = tensor(H, H)

psi_0 = H_H @ psi_initial

print("After H ⊗ H:")
print_state(psi_0)

After H ⊗ H:
(0.500+0.000j)|00> + (-0.500+0.000j)|01> + (0.500+0.000j)|10> + (-0.500+0.000j)|11>


## Apply oracle
We apply $f$ on above state in terms of unitory operator.

In [15]:
def apply_oracle(psi, f):
    U_f = deutsch_oracle(f)
    return U_f @ psi


# Example: f(x) = x
f = DEUTSCH_FUNCTIONS["identity"]

psi_1 = apply_oracle(psi_0, f)

print("After oracle:")
print_state(psi_1)

After oracle:
(0.500+0.000j)|00> + (-0.500+0.000j)|01> + (-0.500+0.000j)|10> + (0.500+0.000j)|11>


## Final Hadamard

In [16]:
# Apply H ⊗ I
H_I = tensor(H, I)

psi_2 = H_I @ psi_1

print("Final quantum state:")
print_state(psi_2)

print("\nMeasurement probabilities:")
print_probabilities(psi_2)

Final quantum state:
(0.707+0.000j)|10> + (-0.707+0.000j)|11>

Measurement probabilities:
|10> : 0.5000
|11> : 0.5000


## Classification

In [17]:
def classify_result(state):

    probs = probabilities(state)

    # First qubit = 0: indices 0 and 1
    p0 = probs[0] + probs[1]

    # First qubit = 1: indices 2 and 3
    p1 = probs[2] + probs[3]

    if np.isclose(p0, 1):
        return "CONSTANT"

    elif np.isclose(p1, 1):
        return "BALANCED"

    return "INCONCLUSIVE"


print("Classification:", classify_result(psi_2))

Classification: BALANCED


In [18]:
# Test all four functions

for name, f in DEUTSCH_FUNCTIONS.items():

    psi_0 = H_H @ psi_initial

    psi_1 = deutsch_oracle(f) @ psi_0

    psi_2 = H_I @ psi_1

    print(f"\nFunction: {name}")
    print("Classification:", classify_result(psi_2))
    print("Probabilities:", probabilities(psi_2))


Function: constant_zero
Classification: CONSTANT
Probabilities: [5.00000000e-01 5.00000000e-01 5.28699741e-34 5.28699741e-34]

Function: constant_one
Classification: CONSTANT
Probabilities: [5.00000000e-01 5.00000000e-01 5.28699741e-34 5.28699741e-34]

Function: identity
Classification: BALANCED
Probabilities: [5.28699741e-34 5.28699741e-34 5.00000000e-01 5.00000000e-01]

Function: negation
Classification: BALANCED
Probabilities: [5.28699741e-34 5.28699741e-34 5.00000000e-01 5.00000000e-01]
